In [1]:
from cropspectai import config
from cropspectai.data_analysis import preprocessing
from cropspectai.modeling.train import train_model

import pandas as pd

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


# --------------------------------------------------
# Load data
# --------------------------------------------------

spectral_length = "as7343" # "full" or "1000" or "as7343"
target = "cab" # "cab" or None
indices = "all" # "all" or "established" or None

if spectral_length == "as7343":
    if target == None:
        train_path = (
            config.PROCESSED_PARQUET_DATA_SPLITS_PATH
            / "all"
            / "train.parquet"
        )
    elif target == "cab":
        if indices == "all": # all engineered features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_all_features.parquet"
                    )
        elif indices == "established": # only established new features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_established_features.parquet"
                    ) 
        elif indices == None: # no engineered features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train.parquet"
                    )       
elif spectral_length == "full" or "1000":
    train_path = (
        config.RAW_PARQUET_DATA_SPLITS_PATH
        / "train.parquet"
    )

df = pd.read_parquet(train_path)
print("Shape of imported data: ", df.shape)

Shape of imported data:  (1607, 41)


In [2]:
# --------------------------------------------------
# Get feature and target column names
# --------------------------------------------------

X, y = preprocessing.split_dataset_into_X_y(df, target)

targets = y.columns.tolist()

print("Targets:")
print(targets)

features = X.columns.tolist()
print("Features:")
print(features)

Targets:
['cab']
Features:
['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR', 'ci_red_edge_as7343', 'ci_green_as7343', 'gm_nd_as7343', 'red_edge_ratio_as7343', 'datt2_as7343', 'datt1_as7343', 'mcari_as7343', 'tcari_as7343', 'osavi_as7343', 'mcari_osavi_as7343', 'tcari_osavi_as7343', 'mndvi_as7343', 'ndvi_as7343', 'gndvi_as7343', 'sr_as7343', 'green_sr_as7343', 'evi2_as7343', 'rdvi_as7343', 'sipi_as7343', 'psri_as7343', 'red_green_nd', 'red_edge_green_nd', 'nir_green_nd', 'red_edge_slope_proxy', 'green_red_ratio', 'rededge_green_ratio', 'nir_fy_ratio', 'nir_fxl_ratio']


## X (features) as indices only

In [3]:
AS7343_CHANNELS = list(config.AS7343_CHANNELS.keys())


X_index_features = X.drop(
        columns=AS7343_CHANNELS,
        errors="ignore",
    ).copy()

index_features = X_index_features.columns.tolist()

# print("Features:")
print(index_features)

# print(X.shape)

# df = pd.concat([X[AS7343_CHANNELS], y], axis = 1).copy()
# print(df.shape)


['ci_red_edge_as7343', 'ci_green_as7343', 'gm_nd_as7343', 'red_edge_ratio_as7343', 'datt2_as7343', 'datt1_as7343', 'mcari_as7343', 'tcari_as7343', 'osavi_as7343', 'mcari_osavi_as7343', 'tcari_osavi_as7343', 'mndvi_as7343', 'ndvi_as7343', 'gndvi_as7343', 'sr_as7343', 'green_sr_as7343', 'evi2_as7343', 'rdvi_as7343', 'sipi_as7343', 'psri_as7343', 'red_green_nd', 'red_edge_green_nd', 'nir_green_nd', 'red_edge_slope_proxy', 'green_red_ratio', 'rededge_green_ratio', 'nir_fy_ratio', 'nir_fxl_ratio']


## Hyperspectra up to 1000nm only

In [4]:
if spectral_length == "1000":
    print(X.shape)
    X_1000 = X.loc[:, "400":"1000"].copy()
    print(X_1000.shape)
    print(y.shape)

    df = pd.concat([X_1000, y], axis = 1).copy()
    print(df.shape)

In [5]:
# --------------------------------------------------
# Store results
# --------------------------------------------------

results = {}


# --------------------------------------------------
# Run XGBoost independently for every feature and every target
# --------------------------------------------------

for feature in index_features:

    print("\n" + "=" * 60)
    print(f"FEATURE: {feature}")
    print("=" * 60)

    X, y = preprocessing.split_dataset_into_X_y(df, target) # here "df" has multiple indices
    
    X_feature = pd.concat([X[AS7343_CHANNELS], X[[feature]]], axis = 1).copy()

    df_feature = pd.concat([X_feature, y], axis = 1).copy() # build new "df" with only one feature

    #check for duplicates
    df_feature = preprocessing.drop_duplicates(
        df_feature,
        target
    ).copy()

    # if df_feature changed, do this - here default
    X_feature, y = preprocessing.split_dataset_into_X_y(df_feature, target)

    X_feature = X_feature.copy()
    y = y.copy()

    # Train / validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X_feature,
        y,
        test_size=0.20,
        random_state=42,
    )

    # Create a fresh model for every target
    model_xgb = XGBRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=1.0,

        tree_method="hist",

        objective="reg:squarederror",
        eval_metric="rmse",

        n_jobs=-1,
        random_state=42,
    )

    # Train
    pipeline_xgb, metrics = train_model(
        model_xgb,
        X_train,
        y_train,
        X_val,
        y_val,
        scale=False,
    )

    metrics["Samples"] = len(df_feature)
    results[feature] = metrics


FEATURE: ci_red_edge_as7343
(Before) Shape of dataframe: (1607, 14)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 14)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) Shape of dataframe: (1606, 14)

FEATURE: ci_green_as7343
(Before) Shape of dataframe: (1607, 14)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 14)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) Shape of dataframe: (1606, 14)

FEATURE: gm_nd_as7343
(Before) Shape of dataframe: (1607, 14)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 14)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) Shape of dataframe: (1606, 14)

FEATURE: red_edge_ratio_as7343
(Before) Shape of dataframe: (1607, 14)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 14)
Checking for rows without labels
Rows with no labels: 0 with target: c

In [6]:
# --------------------------------------------------
# Create results table
# --------------------------------------------------

results_df = pd.DataFrame(results)
print("\nXGBoost Results for each target")
print(results_df)


XGBoost Results for each target
                  ci_red_edge_as7343  ci_green_as7343  gm_nd_as7343  \
MAE (train)                 2.738662         2.656893      2.630521   
MAPE (train) [%]            8.579516         8.292966      8.291070   
MAE (test)                  6.912372         6.788457      7.099106   
MAPE (test) [%]            20.284311        19.595727     20.835729   
R² (test)                   0.537348         0.546906      0.521365   
Samples                  1606.000000      1606.000000   1606.000000   

                  red_edge_ratio_as7343  datt2_as7343  datt1_as7343  \
MAE (train)                    2.630521      2.639518      2.549499   
MAPE (train) [%]               8.291070      8.336568      7.985190   
MAE (test)                     7.099106      6.956626      6.794240   
MAPE (test) [%]               20.835729     20.405550     20.088926   
R² (test)                      0.521365      0.529770      0.553553   
Samples                     1606.000000   1

In [7]:
results_df.head()

,ci_red_edge_as7343,ci_green_as7343,gm_nd_as7343,red_edge_ratio_as7343,datt2_as7343,datt1_as7343,mcari_as7343,tcari_as7343,osavi_as7343,mcari_osavi_as7343,...,sipi_as7343,psri_as7343,red_green_nd,red_edge_green_nd,nir_green_nd,red_edge_slope_proxy,green_red_ratio,rededge_green_ratio,nir_fy_ratio,nir_fxl_ratio
MAE (train),2.738662,2.656893,2.630521,2.630521,2.639518,2.549499,2.691577,2.709275,2.675166,2.652813,...,2.555255,2.498121,2.637238,2.620782,2.656893,2.755900,2.631521,2.620782,2.665601,2.677809
MAPE (train) [%],8.579516,8.292966,8.291070,8.291070,8.336568,7.985190,8.426207,8.464192,8.451677,8.288877,...,8.098217,7.860711,8.175898,8.158553,8.292966,8.593738,8.175806,8.158553,8.322052,8.361389
MAE (test),6.912372,6.788457,7.099106,7.099106,6.956626,6.794240,6.787861,6.742255,7.067597,6.782841,...,7.083797,6.848914,6.587058,6.653439,6.788457,6.769901,6.583683,6.653439,6.855211,6.826123
MAPE (test) [%],20.284311,19.595727,20.835729,20.835729,20.405550,20.088926,19.850996,19.880757,20.825285,19.885069,...,20.899598,20.502235,19.151375,19.415326,19.595727,19.899762,19.079402,19.415326,19.950777,19.852309
R² (test),0.537348,0.546906,0.521365,0.521365,0.529770,0.553553,0.549536,0.550138,0.532007,0.552944,...,0.517152,0.556745,0.570407,0.572201,0.546906,0.553091,0.569194,0.572201,0.544402,0.553728


In [8]:
results_df.iloc[-2, :].sort_values(ascending=False)

red_edge_green_nd        0.572201
rededge_green_ratio      0.572201
red_green_nd             0.570407
green_red_ratio          0.569194
psri_as7343              0.556745
nir_fxl_ratio            0.553728
datt1_as7343             0.553553
red_edge_slope_proxy     0.553091
mcari_osavi_as7343       0.552944
tcari_as7343             0.550138
mcari_as7343             0.549536
evi2_as7343              0.548098
nir_green_nd             0.546906
gndvi_as7343             0.546906
ci_green_as7343          0.546906
green_sr_as7343          0.546906
tcari_osavi_as7343       0.545729
nir_fy_ratio             0.544402
rdvi_as7343              0.542228
ci_red_edge_as7343       0.537348
osavi_as7343             0.532007
ndvi_as7343              0.529770
datt2_as7343             0.529770
sr_as7343                0.529770
red_edge_ratio_as7343    0.521365
gm_nd_as7343             0.521365
sipi_as7343              0.517152
mndvi_as7343             0.516477
Name: R² (test), dtype: float64

In [11]:
top_4_indices = (
    results_df.iloc[-2, :]
    .sort_values(ascending=False)
    .head(4)
    .index
    .tolist()
)

print("top 4 indices in terms of R2 score: ", top_4_indices)

top 4 indices in terms of R2 score:  ['red_edge_green_nd', 'rededge_green_ratio', 'red_green_nd', 'green_red_ratio']


In [13]:
results_df.loc["MAPE (train) [%]", :].sort_values(ascending=True)

psri_as7343              7.860711
datt1_as7343             7.985190
sipi_as7343              8.098217
red_edge_green_nd        8.158553
rededge_green_ratio      8.158553
green_red_ratio          8.175806
red_green_nd             8.175898
mndvi_as7343             8.196117
mcari_osavi_as7343       8.288877
red_edge_ratio_as7343    8.291070
gm_nd_as7343             8.291070
green_sr_as7343          8.292966
ci_green_as7343          8.292966
nir_green_nd             8.292966
gndvi_as7343             8.292966
nir_fy_ratio             8.322052
datt2_as7343             8.336568
ndvi_as7343              8.336568
sr_as7343                8.336568
nir_fxl_ratio            8.361389
mcari_as7343             8.426207
osavi_as7343             8.451677
tcari_as7343             8.464192
evi2_as7343              8.473178
rdvi_as7343              8.473872
tcari_osavi_as7343       8.544708
ci_red_edge_as7343       8.579516
red_edge_slope_proxy     8.593738
Name: MAPE (train) [%], dtype: float64

In [14]:
results_df.loc["MAPE (test) [%]", :].sort_values(ascending=True)

green_red_ratio          19.079402
red_green_nd             19.151375
rededge_green_ratio      19.415326
red_edge_green_nd        19.415326
gndvi_as7343             19.595727
ci_green_as7343          19.595727
nir_green_nd             19.595727
green_sr_as7343          19.595727
mcari_as7343             19.850996
nir_fxl_ratio            19.852309
tcari_as7343             19.880757
mcari_osavi_as7343       19.885069
red_edge_slope_proxy     19.899762
nir_fy_ratio             19.950777
rdvi_as7343              20.001936
datt1_as7343             20.088926
evi2_as7343              20.240290
ci_red_edge_as7343       20.284311
ndvi_as7343              20.405550
sr_as7343                20.405550
datt2_as7343             20.405550
psri_as7343              20.502235
tcari_osavi_as7343       20.510957
osavi_as7343             20.825285
red_edge_ratio_as7343    20.835729
gm_nd_as7343             20.835729
sipi_as7343              20.899598
mndvi_as7343             21.023431
Name: MAPE (test) [%

In [10]:
# file_name = f"xgboost_results_{target}_{spectral_length}-{indices}-only.csv"
# output_path = config.REPORTS_DIR / file_name

# results_df.to_csv(
#     output_path,
#     index=True,
# )

- full length hyperspectra gives better R2 score for all targets except CAB
- 1000nm cutoff hyperspectra gives consistently lower R2 score than full except for CAB
- as7343 does poor at all targets and consistently lower than 1000nm 
 - but CAB score remains on par with 1000nm
 - more importantly does better than even full

Conclusion:
- CAB is better predicted with data within 1000nm range
- as7343 is better capable at predicting CAB than all other targets